In [12]:
!cd /kaggle/working/ViDubb && /kaggle/working/venv/bin/python -c "import os;from dotenv import load_dotenv;load_dotenv();from huggingface_hub import whoami;w=whoami(token=os.getenv('HF_TOKEN'));print('user:',w['name'])"!cd /kaggle/working/ViDubb && /kaggle/working/venv/bin/python inference.py --video_url videos/origin1.mp4 --source_language fr --target_language en --whisper_model small --LipSync False --Bg_sound False

user: anjiall


In [13]:
!cd /kaggle/working/ViDubb && /kaggle/working/venv/bin/python inference.py --video_url videos/origin1.mp4 --source_language fr --target_language en --whisper_model small --LipSync False --Bg_sound False

                                                                                                                        
                                                                                                                        
                                                                                                                        
                                  rvxQ                                                                                  
           {^::_''';v     >,__'':'_'_'^l 2=:-----`-````...-;i.                c^:---_:=I     >;'---':"g                 
           )_.i?II/.:c  z+-_%III"^?t]/.:;;`_yhghd44dddSpJtc/``^l              |`_fhht _c    Q;."qhhi ,d                 
           j^.;1Lj1,.,7%,.;?jLe<`^I]I+````.=DQQ0SFpppqEkBQQH4['-=;...:^v1+,..`'.+DQQ1  ` .`,l' ?QQR) .` .-^I            
            t;.^1oo*_`:_.)1oo*;.`"ii\````` {00QY.      `^shQQQg=.=Lun\._:`;jnnv {00M?1fF2TI/.` wQQAczyF3Lr;`;c          
             s:."1tt%`._%ta1i```

In [14]:
import pathlib
p = pathlib.Path("/kaggle/working/ViDubb/inference.py"); s = p.read_text()
STUB = '''
class _OneSpeaker:
    def __call__(self, audio_file):
        from pydub import AudioSegment as _AS
        dur = len(_AS.from_file(audio_file)) / 1000.0
        class _Seg:
            start, end = 0.0, dur
        class _Ann:
            def itertracks(self, yield_label=False):
                yield _Seg(), None, "SPEAKER_00"
        return _Ann()
'''
if "_OneSpeaker" not in s:
    s = s.replace("\nclass VideoDubbing:", STUB + "\nclass VideoDubbing:", 1)
    s = s.replace('"pyannote/speaker-diarization"', '"pyannote/speaker-diarization-3.1"')
    s = s.replace("use_auth_token=self.huggingface_auth_token).to(device)",
                  "use_auth_token=self.huggingface_auth_token) or _OneSpeaker()", 1)
    p.write_text(s)
t = p.read_text()
print("patched:", "_OneSpeaker" in t, "| 3.1:", "diarization-3.1" in t, "| line107:", [l.strip() for l in t.splitlines() if "use_auth_token=self.huggingface" in l])

patched: True | 3.1: True | line107: ['use_auth_token=self.huggingface_auth_token) or _OneSpeaker()']


In [ ]:
!cd /kaggle/working/ViDubb && /kaggle/working/venv/bin/python inference.py --video_url videos/origin1.mp4 --source_language fr --target_language en --whisper_model small --LipSync False --Bg_sound False

                                                                                                                        
                                                                                                                        
                                                                                                                        
                                  rvxQ                                                                                  
           {^::_''';v     >,__'':'_'_'^l 2=:-----`-````...-;i.                c^:---_:=I     >;'---':"g                 
           )_.i?II/.:c  z+-_%III"^?t]/.:;;`_yhghd44dddSpJtc/``^l              |`_fhht _c    Q;."qhhi ,d                 
           j^.;1Lj1,.,7%,.;?jLe<`^I]I+````.=DQQ0SFpppqEkBQQH4['-=;...:^v1+,..`'.+DQQ1  ` .`,l' ?QQR) .` .-^I            
            t;.^1oo*_`:_.)1oo*;.`"ii\````` {00QY.      `^shQQQg=.=Lun\._:`;jnnv {00M?1fF2TI/.` wQQAczyF3Lr;`;c          
             s:."1tt%`._%ta1i```

In [1]:
!ls /kaggle/working/venv/bin/python 2>/dev/null && echo VENV_OK || echo VENV_GONE
!ls -lh /kaggle/working/ViDubb/results/ 2>/dev/null || echo "no results dir"

VENV_GONE
no results dir


In [2]:
!bash /kaggle/working/setup.sh

bash: /kaggle/working/setup.sh: No such file or directory


In [3]:
import pathlib, subprocess
script = r'''#!/bin/bash
set -e
pip install -q uv
uv python install 3.11
rm -rf /kaggle/working/venv
uv venv --python 3.11 --seed /kaggle/working/venv
P=/kaggle/working/venv/bin/python
uv pip install --python $P torch==2.5.1 torchvision==0.20.1 torchaudio==2.5.1 --index-url https://download.pytorch.org/whl/cu121
cd /kaggle/working
rm -rf ViDubb
git clone --depth 1 https://github.com/medahmedkrichen/ViDubb.git
cd ViDubb
grep -v -E '^(tensorflow|nltk)==' requirements.txt > req.kaggle.txt
uv pip install --python $P -r req.kaggle.txt
uv pip install --python $P ipython "huggingface_hub==0.26.2"
uv pip install --python $P --no-deps TTS==0.21.0
uv pip install --python $P tensorflow-cpu==2.17.1 tf-keras deepface==0.0.93 spacy==3.8.2
uv pip install --python $P "setuptools<81" wheel
uv pip install --python $P --no-build-isolation openai-whisper==20240930
uv pip install --python $P "numpy==1.26.4"
wget -q 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth' -O Wav2Lip/wav2lip_gan.pth
wget -q 'https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth' -O Wav2Lip/face_detection/detection/sfd/s3fd.pth
$P -c "import torch,numpy;print('ENV OK',torch.__version__,torch.cuda.is_available(),numpy.__version__)"
'''
pathlib.Path("/kaggle/working/setup.sh").write_text(script)
print("script written:", len(script), "bytes")
subprocess.run(["bash", "/kaggle/working/setup.sh"], check=True)

script written: 1270 bytes


Installed Python 3.11.18 in 1.24s
 + cpython-3.11.18-linux-x86_64-gnu (python3.11)
Using CPython 3.11.18
Creating virtual environment with seed packages at: venv
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
 + packaging==26.4
 + pip==26.3
 + setuptools==84.2.1
 + wheel==0.48.0
Activate with: source venv/bin/activate
Using Python 3.11.18 environment at: venv
Resolved 26 packages in 611ms
Prepared 26 packages in 30.64s
         If the cache and target directories are on different filesystems, hardlinking may not be supported.
         If this is intentional, set `export UV_LINK_MODE=copy` or use `--link-mode=copy` to suppress this warning.
Installed 26 packages in 6.07s
 + filelock==4.2.4
 + fsspec==2026.9.0
 + jinja2==3.1.6
 + markupsafe==3.0.3
 + mpmath==1.3.0
 + networkx==3.6.1
 + numpy==2.4.6
 + nvidia-cubl

ENV OK 2.5.1+cu121 True 1.26.4


CompletedProcess(args=['bash', '/kaggle/working/setup.sh'], returncode=0)

In [4]:
from kaggle_secrets import UserSecretsClient
hf = UserSecretsClient().get_secret("kagglevidubb2")
open("/kaggle/working/ViDubb/.env","w").write(f'HF_TOKEN = "{hf}"\nGroq_TOKEN = ""\n')
print("len:", len(hf))


len: 37


In [5]:
import pathlib
p = pathlib.Path("/kaggle/working/ViDubb/inference.py"); s = p.read_text()
STUB = '''
class _OneSpeaker:
    def __call__(self, audio_file):
        from pydub import AudioSegment as _AS
        dur = len(_AS.from_file(audio_file)) / 1000.0
        class _Seg:
            start, end = 0.0, dur
        class _Ann:
            def itertracks(self, yield_label=False):
                yield _Seg(), None, "SPEAKER_00"
        return _Ann()
'''
if "_OneSpeaker" not in s:
    s = s.replace("\nclass VideoDubbing:", STUB + "\nclass VideoDubbing:", 1)
    s = s.replace('"pyannote/speaker-diarization"', '"pyannote/speaker-diarization-3.1"')
    s = s.replace("use_auth_token=self.huggingface_auth_token).to(device)",
                  "use_auth_token=self.huggingface_auth_token) or _OneSpeaker()", 1)
    p.write_text(s)
t = p.read_text(); print("patched:", "_OneSpeaker" in t, "| 3.1:", "diarization-3.1" in t)

patched: True | 3.1: True


In [6]:
!cd /kaggle/working/ViDubb && ffmpeg -y -loglevel error -ss 00:00:10 -t 40 -i videos/origin1.mp4 -c copy /kaggle/working/clip40.mp4 && ls -lh /kaggle/working/clip40.mp4

-rw-r--r-- 1 root root 1.9M Nov 26 06:26 /kaggle/working/clip40.mp4


In [7]:
!cd /kaggle/working/ViDubb && /kaggle/working/venv/bin/python inference.py --video_url /kaggle/working/clip40.mp4 --source_language fr --target_language en --whisper_model base --LipSync False --Bg_sound False

                                                                                                                        
                                                                                                                        
                                                                                                                        
                                  rvxQ                                                                                  
           {^::_''';v     >,__'':'_'_'^l 2=:-----`-````...-;i.                c^:---_:=I     >;'---':"g                 
           )_.i?II/.:c  z+-_%III"^?t]/.:;;`_yhghd44dddSpJtc/``^l              |`_fhht _c    Q;."qhhi ,d                 
           j^.;1Lj1,.,7%,.;?jLe<`^I]I+````.=DQQ0SFpppqEkBQQH4['-=;...:^v1+,..`'.+DQQ1  ` .`,l' ?QQR) .` .-^I            
            t;.^1oo*_`:_.)1oo*;.`"ii\````` {00QY.      `^shQQQg=.=Lun\._:`;jnnv {00M?1fF2TI/.` wQQAczyF3Lr;`;c          
             s:."1tt%`._%ta1i```

In [8]:
!cd /kaggle/working/ViDubb && TF_USE_LEGACY_KERAS=1 /kaggle/working/venv/bin/python inference.py --video_url /kaggle/working/clip40.mp4 --source_language fr --target_language en --whisper_model base

                                                                                                                        
                                                                                                                        
                                                                                                                        
                                  rvxQ                                                                                  
           {^::_''';v     >,__'':'_'_'^l 2=:-----`-````...-;i.                c^:---_:=I     >;'---':"g                 
           )_.i?II/.:c  z+-_%III"^?t]/.:;;`_yhghd44dddSpJtc/``^l              |`_fhht _c    Q;."qhhi ,d                 
           j^.;1Lj1,.,7%,.;?jLe<`^I]I+````.=DQQ0SFpppqEkBQQH4['-=;...:^v1+,..`'.+DQQ1  ` .`,l' ?QQR) .` .-^I            
            t;.^1oo*_`:_.)1oo*;.`"ii\````` {00QY.      `^shQQQg=.=Lun\._:`;jnnv {00M?1fF2TI/.` wQQAczyF3Lr;`;c          
             s:."1tt%`._%ta1i```

In [9]:
import os, sys, subprocess, json, urllib.request

print("=" * 50)
print("1. Kaggle environment")
print("=" * 50)
print("KAGGLE_KERNEL_RUN_TYPE:", os.environ.get("KAGGLE_KERNEL_RUN_TYPE", "not set"))
print("Working dir:", os.getcwd())

# Check whether GPU is actually attached (the UI switch being on doesn't mean it's actually attached)
print("\n--- nvidia-smi ---")
r = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                   capture_output=True, text=True)
print(r.stdout or r.stderr)

# Check whether torch can see CUDA
try:
    import torch
    print("torch.cuda.is_available():", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("Device:", torch.cuda.get_device_name(0))
except Exception as e:
    print("torch import failed:", e)

print("\n" + "=" * 50)
print("2. GitHub reachability (git clone dry run)")
print("=" * 50)
r = subprocess.run(["git", "ls-remote", "--heads", "https://github.com/medahmedkrichen/ViDubb.git"],
                   capture_output=True, text=True, timeout=30)
print("returncode:", r.returncode)
if r.returncode == 0:
    print("GitHub OK, heads:", r.stdout.strip()[:200])
else:
    print("GitHub FAILED:", r.stderr.strip()[:300])

print("\n" + "=" * 50)
print("3. Hugging Face token + gated access")
print("=" * 50)
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN from Kaggle Secrets:", "present" if HF_TOKEN else "EMPTY")
except Exception as e:
    print("Kaggle Secrets read failed:", e)
    HF_TOKEN = None

if HF_TOKEN:
    req = urllib.request.Request("https://huggingface.co/api/whoami-v2",
                                 headers={"Authorization": f"Bearer {HF_TOKEN}"})
    try:
        with urllib.request.urlopen(req, timeout=15) as resp:
            data = json.loads(resp.read())
            print("whoami name:", data.get("name"))
            auth = data.get("auth", {})
            access = auth.get("accessToken", {})
            print("token role:", access.get("role"))
            print("canReadGatedRepos:", access.get("fineGrained", {}).get("canReadGatedRepos", "N/A"))
    except Exception as e:
        print("whoami failed (token invalid or network issue):", e)

1. Kaggle environment
KAGGLE_KERNEL_RUN_TYPE: Interactive
Working dir: /kaggle/working

--- nvidia-smi ---
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB

torch.cuda.is_available(): True
Device: Tesla T4

2. GitHub reachability (git clone dry run)
returncode: 0
GitHub OK, heads: 6a90ea25fd14d7cb241f1c5759fc9f4b8b93f49d	refs/heads/main

3. Hugging Face token + gated access
HF_TOKEN from Kaggle Secrets: present
whoami failed (token invalid or network issue): HTTP Error 401: Unauthorized


In [10]:
from kaggle_secrets import UserSecretsClient
import urllib.request, json

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
print("Token prefix:", HF_TOKEN[:8] + "..." if HF_TOKEN else "EMPTY")

req = urllib.request.Request("https://huggingface.co/api/whoami-v2",
                             headers={"Authorization": f"Bearer {HF_TOKEN}"})
try:
    with urllib.request.urlopen(req, timeout=15) as resp:
        data = json.loads(resp.read())
        print("whoami OK:", data.get("name"))
except Exception as e:
    print("Still failing:", e)

Token prefix: hf_EVXgk...
Still failing: HTTP Error 401: Unauthorized


In [1]:
import urllib.request, json
from kaggle_secrets import UserSecretsClient

HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# 1. Shape check — HF user tokens start with "hf_" and are ~37 chars, no whitespace
print("len:", len(HF_TOKEN) if HF_TOKEN else 0)
print("starts hf_ :", HF_TOKEN.startswith("hf_") if HF_TOKEN else False)
print("has whitespace:", any(c.isspace() for c in HF_TOKEN) if HF_TOKEN else "n/a")
print("repr(first 10):", repr(HF_TOKEN[:10]) if HF_TOKEN else "n/a")
print("repr(last 4):", repr(HF_TOKEN[-4:]) if HF_TOKEN else "n/a")

# 2. whoami — with Authorization header explicitly built
req = urllib.request.Request(
    "https://huggingface.co/api/whoami-v2",
    headers={"Authorization": f"Bearer {HF_TOKEN.strip()}" if HF_TOKEN else ""}
)
try:
    with urllib.request.urlopen(req, timeout=15) as r:
        print("whoami OK:", json.loads(r.read())["name"])
except urllib.error.HTTPError as e:
    print("whoami HTTP", e.code, e.reason)

# 3. Gated repo probe — does this token see the gated model?
req2 = urllib.request.Request(
    "https://huggingface.co/api/models/pyannote/speaker-diarization-3.1",
    headers={"Authorization": f"Bearer {HF_TOKEN.strip()}" if HF_TOKEN else ""}
)
try:
    with urllib.request.urlopen(req2, timeout=15) as r:
        print("gated repo visible: YES")
except urllib.error.HTTPError as e:
    print("gated repo HTTP", e.code, e.reason)

len: 37
starts hf_ : True
has whitespace: False
repr(first 10): 'hf_EVXgkTk'
repr(last 4): 'kWWK'
whoami HTTP 401 Unauthorized
gated repo visible: YES


In [2]:
import os
import subprocess

# 1. Clone ViDubb (using --depth 1 for speed)
if not os.path.exists("ViDubb"):
    print("Cloning ViDubb...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/medahmedkrichen/ViDubb.git"], check=True)

# 2. Write the .env file using the token we just verified
from kaggle_secrets import UserSecretsClient
HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")

# We write the .env file exactly as the repo expects
with open("ViDubb/.env", "w") as f:
    f.write(f"HF_TOKEN = '{HF_TOKEN}'\n")
    f.write("Groq_TOKEN = ''\n")

print("Clone and .env setup complete.")
print("Contents of .env (masked):")
with open("ViDubb/.env") as f:
    for line in f:
        if "HF_TOKEN" in line:
            print("  HF_TOKEN = 'hf_***'")
        else:
            print(f"  {line.strip()}")

Cloning ViDubb...


Cloning into 'ViDubb'...


Clone and .env setup complete.
Contents of .env (masked):
  HF_TOKEN = 'hf_***'
  Groq_TOKEN = ''


In [3]:
import sys

# We install only what ViDubb needs, avoiding the broken pins in requirements.txt
essential_packages = [
    "pyannote.audio==3.3.2",
    "python-dotenv==1.0.1",
    "speechbrain==1.0.2",
    "audio-separator==0.24.1",
    "onnxruntime==1.20.1",
    "yt-dlp",
    "groq==0.13.0",
    "ffmpeg-python==0.2.0",
    "faster-whisper==1.1.1",
    "transformers==4.46.3",
    "librosa==0.10.2.post1",
    "gradio==5.8.0",
    "protobuf==3.19.6",
    "spacy==3.8.2",
    "packaging==20.9",
    "numpy==1.26.4"
]

print("Installing essential packages (this takes ~3-5 mins)...")
!pip install -q {" ".join(essential_packages)}
print("Installs complete.")

Installing essential packages (this takes ~3-5 mins)...
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  P

In [9]:
import os
import subprocess
import sys
import importlib

# ==============================================================================
# 1. SETUP SECRETS & CLONE REPO
# ==============================================================================
print("--- Step 1: Setup and Clone ---")
from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN retrieved from Kaggle Secrets.")
except Exception as e:
    raise RuntimeError("Failed to get HF_TOKEN from Kaggle Secrets. Check Add-ons -> Secrets.") from e

if not os.path.exists("ViDubb"):
    print("Cloning ViDubb repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/medahmedkrichen/ViDubb.git"], check=True)
else:
    print("ViDubb already cloned.")

# Write the .env file
with open("ViDubb/.env", "w") as f:
    f.write(f"HF_TOKEN = '{HF_TOKEN}'\n")
    f.write("Groq_TOKEN = ''\n")
print("ViDubb/.env configured.")

# ==============================================================================
# 2. INSTALL DEPENDENCIES (Fixing the protobuf/onnx conflict)
# ==============================================================================
print("\n--- Step 2: Install Dependencies ---")
# We install in chunks to prevent pip's resolver from deadlocking on protobuf
packages = [
    "numpy==1.26.4", "packaging==20.9", "python-dotenv==1.0.1",
    "faster-whisper==1.1.1", "transformers==4.46.3", "librosa==0.10.2.post1",
    "pyannote.audio==3.3.2", "speechbrain==1.0.2", "audio-separator==0.24.1",
    "onnxruntime==1.20.1", "yt-dlp", "groq==0.13.0", "ffmpeg-python==0.2.0",
    "gradio==5.8.0", "spacy==3.8.2"
]
print("Installing core packages...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + packages, check=True)

print("Installing runtime patches...")
# Inference.py expects these, but installing them upfront avoids mid-run crashes
# We DO NOT pin protobuf here so onnx/audio-separator resolve their own compatible version.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "TTS==0.21.0", "deepface==0.0.93", "openai-whisper==20240930"], check=True)
print("Dependencies installed.")

# ==============================================================================
# 3. DOWNLOAD WAV2LIP WEIGHTS
# ==============================================================================
print("\n--- Step 3: Downloading Wav2Lip Weights ---")
wav2lip_path = "ViDubb/Wav2Lip/wav2lip_gan.pth"
s3fd_path = "ViDubb/Wav2Lip/face_detection/detection/sfd/s3fd.pth"

if not os.path.exists(wav2lip_path) or os.path.getsize(wav2lip_path) < 1000000:
    print("Downloading wav2lip_gan.pth...")
    subprocess.run(["wget", "-q", "https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth", "-O", wav2lip_path], check=True)

if not os.path.exists(s3fd_path) or os.path.getsize(s3fd_path) < 1000000:
    print("Downloading s3fd.pth...")
    subprocess.run(["wget", "-q", "https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth", "-O", s3fd_path], check=True)
    
print("Weights downloaded successfully.")

# ==============================================================================
# 4. RUN INFERENCE PIPELINE
# ==============================================================================
print("\n--- Step 4: Running Dubbing Pipeline ---")
# ⚠️ CHANGE THIS to point to your actual uploaded video in /kaggle/input/ ⚠️
VIDEO_URL = "/kaggle/input/video2translate/dummy_video.mp4" 

if not os.path.exists(VIDEO_URL):
    print(f"⚠️ WARNING: Video not found at {VIDEO_URL}")
    print("Skipping inference. Please upload a video and update VIDEO_URL.")
else:
    os.chdir("ViDubb")
    # We run the script using python, passing the arguments.
    # Source and target languages can be changed (e.g., "en" -> "fr")
    # LipSync True requires the weights we just downloaded.
    cmd = [
        sys.executable, "inference.py",
        "--video_url", VIDEO_URL,
        "--source_language", "en",
        "--target_language", "fr",
        "--LipSync", "True",
        "--Bg_sound", "True"
    ]
    
    print(f"Executing: {' '.join(cmd)}")
    try:
        # stream output directly to Kaggle cell
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Pipeline finished successfully!")
            print("Check ViDubb/results/ for your dubbed video.")
        else:
            print(f"\n❌ Pipeline failed with exit code {process.returncode}")
    except Exception as e:
        print(f"\n❌ Execution error: {e}")
    finally:
        os.chdir("..")

--- Step 1: Setup and Clone ---
HF_TOKEN retrieved from Kaggle Secrets.
ViDubb already cloned.
ViDubb/.env configured.

--- Step 2: Install Dependencies ---
Installing core packages...


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'matplotlib' when getting requirements to build wheel


CalledProcessError: Command '['/usr/bin/python3', '-m', 'pip', 'install', '-q', 'numpy==1.26.4', 'packaging==20.9', 'python-dotenv==1.0.1', 'faster-whisper==1.1.1', 'transformers==4.46.3', 'librosa==0.10.2.post1', 'pyannote.audio==3.3.2', 'speechbrain==1.0.2', 'audio-separator==0.24.1', 'onnxruntime==1.20.1', 'yt-dlp', 'groq==0.13.0', 'ffmpeg-python==0.2.0', 'gradio==5.8.0', 'spacy==3.8.2']' returned non-zero exit status 1.

In [10]:
import os
import subprocess
import sys

# ==============================================================================
# 1. SETUP SECRETS & CLONE REPO
# ==============================================================================
print("--- Step 1: Setup and Clone ---")
from kaggle_secrets import UserSecretsClient
try:
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN retrieved from Kaggle Secrets.")
except Exception as e:
    raise RuntimeError("Failed to get HF_TOKEN from Kaggle Secrets.") from e

if not os.path.exists("ViDubb"):
    print("Cloning ViDubb repository...")
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/medahmedkrichen/ViDubb.git"], check=True)
else:
    print("ViDubb already cloned.")

with open("ViDubb/.env", "w") as f:
    f.write(f"HF_TOKEN = '{HF_TOKEN}'\n")
    f.write("Groq_TOKEN = ''\n")
print("ViDubb/.env configured.")

# ==============================================================================
# 2. INSTALL DEPENDENCIES (Force wheels, no source compilation)
# ==============================================================================
print("\n--- Step 2: Install Dependencies (force binary wheels) ---")

# Core packages: force pre-built wheels only, avoid source compilation
core_packages = [
    "numpy==1.26.4", "packaging==20.9", "python-dotenv==1.0.1",
    "faster-whisper==1.1.1", "transformers==4.46.3", "librosa==0.10.2.post1",
    "pyannote.audio==3.3.2", "speechbrain==1.0.2", "audio-separator==0.24.1",
    "onnxruntime==1.20.1", "yt-dlp", "groq==0.13.0", "ffmpeg-python==0.2.0",
    "gradio==5.8.0", "spacy==3.8.2"
]

print("Installing core packages with --only-binary :all: ...")
# Key: --only-binary :all: forces pip to download wheels only; if a package has no wheel, it will skip instead of compiling
r = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--only-binary", ":all:"] + core_packages,
    check=False  # Do not interrupt even if a package fails
)
if r.returncode != 0:
    print("⚠️ Some core packages failed to install with wheel-only. Retrying without restriction for those...")
    # If the forced wheel version fails, retry once allowing compilation (for packages that truly lack wheels)
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q"] + core_packages,
        check=False
    )

print("Installing runtime patches...")
# DeepFace/Whisper/TTS also force wheels to prevent cascading compilation failures
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "--only-binary", ":all:",
     "TTS==0.21.0", "deepface==0.0.93", "openai-whisper==20240930"],
    check=False
)
print("Dependencies installed.")

# ==============================================================================
# 3. DOWNLOAD WAV2LIP WEIGHTS
# ==============================================================================
print("\n--- Step 3: Downloading Wav2Lip Weights ---")
wav2lip_path = "ViDubb/Wav2Lip/wav2lip_gan.pth"
s3fd_path = "ViDubb/Wav2Lip/face_detection/detection/sfd/s3fd.pth"

if not os.path.exists(wav2lip_path) or os.path.getsize(wav2lip_path) < 1000000:
    print("Downloading wav2lip_gan.pth...")
    subprocess.run(["wget", "-q", "https://github.com/medahmedkrichen/ViDubb/releases/download/weights2/wav2lip_gan.1.1.pth", "-O", wav2lip_path], check=True)

if not os.path.exists(s3fd_path) or os.path.getsize(s3fd_path) < 1000000:
    print("Downloading s3fd.pth...")
    subprocess.run(["wget", "-q", "https://github.com/medahmedkrichen/ViDubb/releases/download/weights1/s3fd-619a316812.1.1.pth", "-O", s3fd_path], check=True)
    
print("Weights downloaded successfully.")

# ==============================================================================
# 4. RUN INFERENCE PIPELINE
# ==============================================================================
print("\n--- Step 4: Running Dubbing Pipeline ---")
VIDEO_URL = "/kaggle/input/video2translate/dummy_video.mp4"  # ⚠️ change to your actual video path

if not os.path.exists(VIDEO_URL):
    print(f"⚠️ WARNING: Video not found at {VIDEO_URL}")
    print("Skipping inference. Please upload a video and update VIDEO_URL.")
else:
    os.chdir("ViDubb")
    cmd = [
        sys.executable, "inference.py",
        "--video_url", VIDEO_URL,
        "--source_language", "en",
        "--target_language", "fr",
        "--LipSync", "True",
        "--Bg_sound", "True"
    ]
    
    print(f"Executing: {' '.join(cmd)}")
    try:
        process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
        process.wait()
        
        if process.returncode == 0:
            print("\n✅ Pipeline finished successfully!")
            print("Check ViDubb/results/ for your dubbed video.")
        else:
            print(f"\n❌ Pipeline failed with exit code {process.returncode}")
    except Exception as e:
        print(f"\n❌ Execution error: {e}")
    finally:
        os.chdir("..")

--- Step 1: Setup and Clone ---
HF_TOKEN retrieved from Kaggle Secrets.
ViDubb already cloned.
ViDubb/.env configured.

--- Step 2: Install Dependencies (force binary wheels) ---
Installing core packages with --only-binary :all: ...


ERROR: Ignored the following yanked versions: 2.4.0
ERROR: Could not find a version that satisfies the requirement numpy==1.26.4 (from versions: 2.1.0, 2.1.1, 2.1.2, 2.1.3, 2.2.0, 2.2.1, 2.2.2, 2.2.3, 2.2.4, 2.2.5, 2.2.6, 2.3.0, 2.3.1, 2.3.2, 2.3.3, 2.3.4, 2.3.5, 2.4.0rc1, 2.4.1, 2.4.2, 2.4.3, 2.4.4, 2.4.5, 2.4.6, 2.5.0rc1, 2.5.0, 2.5.1, 2.5.2, 2.5.3, 2.5.4, 2.5.5)
ERROR: No matching distribution found for numpy==1.26.4


⚠️ Some core packages failed to install with wheel-only. Retrying without restriction for those...


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> No available output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
ERROR: Failed to build 'matplotlib' when getting requirements to build wheel


Installing runtime patches...


ERROR: Could not find a version that satisfies the requirement TTS==0.21.0 (from versions: none)
ERROR: No matching distribution found for TTS==0.21.0


Dependencies installed.

--- Step 3: Downloading Wav2Lip Weights ---
Weights downloaded successfully.

--- Step 4: Running Dubbing Pipeline ---
⚠️ WARNING: Video not found at /kaggle/input/video2translate/dummy_video.mp4
Skipping inference. Please upload a video and update VIDEO_URL.
